In [4]:
# ═══════════════════════════════════════════════════════════
# CELL 1: DATA LOADING & FEATURE EXTRACTION
# ═══════════════════════════════════════════════════════════
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.decomposition import PCA
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter

DATA_PATH = "/kaggle/input/eeg-ds/ds005540"
SUBJECTS = range(1, 55)

emotion_7_map = {
    'score_1': 'joy', 'score_2': 'inspiration', 'score_3': 'tenderness',
    'score_4': 'fear', 'score_5': 'disgust', 'score_6': 'sadness',
    'score_7': 'neutral'
}
sentiment_3_map = {
    'joy': 'positive', 'inspiration': 'positive', 'tenderness': 'positive',
    'fear': 'negative', 'disgust': 'negative', 'sadness': 'negative',
    'neutral': 'neutral'
}

# ─────────────────────────────────────────────────
# FEATURE EXTRACTION
# ─────────────────────────────────────────────────
# 5 frequency bands carry different neural signals:
#   Band 0 = Delta (1-4 Hz)   → deep processing
#   Band 1 = Theta (4-8 Hz)   → emotional processing
#   Band 2 = Alpha (8-14 Hz)  → relaxation (low alpha = engaged)
#   Band 3 = Beta (14-30 Hz)  → stress, alertness
#   Band 4 = Gamma (30-47 Hz) → high-level cognition

def extract_features(trial_eeg):
    """Extract 832 features from one trial (64 ch × 30 timepoints × 5 bands)."""
    n_channels = trial_eeg.shape[0]
    n_bands = trial_eeg.shape[2]
    features = []

    # Per-channel, per-band: mean + std → 64 × 5 × 2 = 640
    for ch in range(n_channels):
        for band in range(n_bands):
            band_data = trial_eeg[ch, :, band]
            features.append(float(np.mean(band_data)))
            features.append(float(np.std(band_data)))

    # Band ratios (neuroscience markers) → 64 × 3 = 192
    for ch in range(n_channels):
        band_means = [float(np.mean(trial_eeg[ch, :, b])) for b in range(n_bands)]
        delta, theta, alpha, beta, gamma = band_means
        eps = 1e-8
        features.append(alpha / (beta + eps))   # Valence marker
        features.append(theta / (beta + eps))   # Engagement marker
        features.append(gamma / (alpha + eps))   # Cognitive load marker

    return features  # 832 per feature type (DE or PSD)


# ─────────────────────────────────────────────────
# LOAD ALL SUBJECTS
# ─────────────────────────────────────────────────
all_X, all_y_7, all_y_3, all_groups = [], [], [], []
skipped = []
loaded = 0

print("Loading EEG data (both sessions, DE + PSD)...\n")

for i in SUBJECTS:
    sub_id = f"sub-{i:02d}"
    beh_path = os.path.join(DATA_PATH, sub_id, "beh",
                            f"{sub_id}_task-emotion_beh.tsv")
    if not os.path.exists(beh_path):
        skipped.append(sub_id)
        continue

    beh_df = pd.read_csv(beh_path, sep='\t')

    sessions = [
        ("ses-ima", 0),    # imagery trials → beh rows 0-20
        ("ses-vid", 21),   # video trials   → beh rows 21-41
    ]

    for ses_name, beh_offset in sessions:
        de_path = os.path.join(DATA_PATH, "derivatives", sub_id, ses_name, "eeg",
                               f"{sub_id}_{ses_name}_task-emotion_de.npy")
        psd_path = os.path.join(DATA_PATH, "derivatives", sub_id, ses_name, "eeg",
                                f"{sub_id}_{ses_name}_task-emotion_psd.npy")

        if not os.path.exists(de_path):
            continue

        de_data = np.load(de_path, allow_pickle=True)
        if hasattr(de_data, 'item'):
            de_data = de_data.item()
        if isinstance(de_data, dict):
            de_data = list(de_data.values())[0]

        has_psd = os.path.exists(psd_path)
        if has_psd:
            psd_data = np.load(psd_path, allow_pickle=True)
            if hasattr(psd_data, 'item'):
                psd_data = psd_data.item()
            if isinstance(psd_data, dict):
                psd_data = list(psd_data.values())[0]

        n_total_time = de_data.shape[1]
        n_trials = 21
        time_per_trial = n_total_time // n_trials

        if i == 1 and ses_name == "ses-ima":
            print(f"  Data shape: {de_data.shape}")
            print(f"  Trials per session: {n_trials}")
            print(f"  Time-windows per trial: {time_per_trial}\n")

        for trial_idx in range(n_trials):
            beh_row = beh_offset + trial_idx
            if beh_row >= len(beh_df):
                break

            t_start = trial_idx * time_per_trial
            t_end = t_start + time_per_trial

            trial_de = de_data[:, t_start:t_end, :]
            features = extract_features(trial_de)  # 832 DE features

            if has_psd:
                trial_psd = psd_data[:, t_start:t_end, :]
                features.extend(extract_features(trial_psd))  # + 832 PSD

            all_X.append(features)

            scores = beh_df.iloc[beh_row][
                ['score_1','score_2','score_3','score_4','score_5','score_6','score_7']
            ].values.astype(float)

            if np.any(np.isnan(scores)):
                all_X.pop()
                continue

            emotion_idx = np.argmax(scores)
            emotion_7 = emotion_7_map[f'score_{emotion_idx+1}']
            all_y_7.append(emotion_7)
            all_y_3.append(sentiment_3_map[emotion_7])
            all_groups.append(i)

    loaded += 1
    print(f"  {sub_id} ✓")

if skipped:
    print(f"\nSkipped {len(skipped)} subjects: {skipped[:5]}...")
print(f"Loaded: {loaded} subjects")

X = np.array(all_X, dtype=np.float64)
y_7 = np.array(all_y_7)
y_3 = np.array(all_y_3)
groups = np.array(all_groups)

bad_mask = np.isnan(X).any(axis=1) | np.isinf(X).any(axis=1)
if bad_mask.sum() > 0:
    print(f"Removing {bad_mask.sum()} rows with NaN/Inf")
    X, y_7, y_3, groups = X[~bad_mask], y_7[~bad_mask], y_3[~bad_mask], groups[~bad_mask]

print(f"\n{'='*50}")
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"{'='*50}")
print(f"\n7-class distribution:")
for e, c in sorted(Counter(y_7).items()):
    print(f"  {e}: {c}")
print(f"\n3-class distribution:")
for s, c in sorted(Counter(y_3).items()):
    print(f"  {s}: {c}")


Loading EEG data (both sessions, DE + PSD)...

  Data shape: (64, 630, 5)
  Trials per session: 21
  Time-windows per trial: 30

  sub-01 ✓
  sub-02 ✓
  sub-03 ✓
  sub-04 ✓
  sub-05 ✓
  sub-06 ✓
  sub-07 ✓
  sub-08 ✓
  sub-09 ✓
  sub-10 ✓
  sub-11 ✓
  sub-12 ✓
  sub-13 ✓
  sub-14 ✓
  sub-15 ✓
  sub-16 ✓
  sub-17 ✓
  sub-18 ✓
  sub-19 ✓
  sub-20 ✓
  sub-21 ✓
  sub-22 ✓
  sub-23 ✓
  sub-24 ✓
  sub-25 ✓
  sub-26 ✓
  sub-27 ✓
  sub-28 ✓
  sub-29 ✓
  sub-30 ✓
  sub-31 ✓
  sub-32 ✓
  sub-33 ✓
  sub-34 ✓
  sub-35 ✓
  sub-36 ✓
  sub-37 ✓
  sub-38 ✓
  sub-39 ✓
  sub-40 ✓
  sub-41 ✓
  sub-42 ✓
  sub-43 ✓
  sub-44 ✓
  sub-45 ✓
  sub-46 ✓
  sub-47 ✓
  sub-48 ✓
  sub-49 ✓
  sub-50 ✓
  sub-51 ✓
  sub-52 ✓
  sub-53 ✓
  sub-54 ✓
Loaded: 54 subjects

Dataset: 2268 samples, 1664 features

7-class distribution:
  disgust: 230
  fear: 235
  inspiration: 361
  joy: 459
  neutral: 427
  sadness: 249
  tenderness: 307

3-class distribution:
  negative: 714
  neutral: 427
  positive: 1127


In [5]:
# ═══════════════════════════════════════════════════════════
# CELL 2: MODEL COMPARISON — 4 models × 2 tasks
# ═══════════════════════════════════════════════════════════
# Added LogReg as a simple baseline and GB for comparison.
# All evaluated with GroupKFold(5) — no subject leaks.

le_7 = LabelEncoder()
le_3 = LabelEncoder()
y_7_encoded = le_7.fit_transform(y_7)
y_3_encoded = le_3.fit_transform(y_3)


def evaluate_model(X, y, groups, model_name):
    """Evaluate a model with GroupKFold(5) cross-validation."""
    gkf = GroupKFold(n_splits=5)
    accuracies, f1_scores_list = [], []

    for train_idx, test_idx in gkf.split(X, y, groups):
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X[train_idx])
        X_test = scaler.transform(X[test_idx])

        if model_name == 'logreg':
            model = LogisticRegression(max_iter=1000, class_weight='balanced',
                                       random_state=42, n_jobs=-1)
        elif model_name == 'svm':
            model = SVC(kernel='rbf', class_weight='balanced',
                        C=1.0, random_state=42)
        elif model_name == 'rf':
            model = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                           random_state=42, n_jobs=-1)
        elif model_name == 'gb':
            model = GradientBoostingClassifier(
                n_estimators=200, max_depth=5, learning_rate=0.1,
                subsample=0.8, max_features='sqrt', random_state=42)
        else:
            raise ValueError(f"Unknown model: {model_name}")

        model.fit(X_train, y[train_idx])
        y_pred = model.predict(X_test)
        accuracies.append(accuracy_score(y[test_idx], y_pred))
        f1_scores_list.append(f1_score(y[test_idx], y_pred, average='weighted'))

    return (np.mean(accuracies), np.std(accuracies),
            np.mean(f1_scores_list), np.std(f1_scores_list))


models = [
    ('logreg', 'Logistic Regression'),
    ('svm',    'SVM (RBF)'),
    ('rf',     'Random Forest'),
    ('gb',     'Gradient Boosting'),
]

print("=" * 60)
print("BASELINE COMPARISON — All 1664 features")
print("=" * 60)

results_7, results_3 = {}, {}

for name, label in models:
    print(f"\n  {label} (7-class)...", end=" ")
    acc, std, f1, f1_std = evaluate_model(X, y_7_encoded, groups, name)
    results_7[name] = {'acc': acc, 'std': std, 'f1': f1}
    print(f"Acc={acc:.3f}±{std:.3f}  F1={f1:.3f}")

    print(f"  {label} (3-class)...", end=" ")
    acc, std, f1, f1_std = evaluate_model(X, y_3_encoded, groups, name)
    results_3[name] = {'acc': acc, 'std': std, 'f1': f1}
    print(f"Acc={acc:.3f}±{std:.3f}  F1={f1:.3f}")

# Summary table
print(f"\n{'='*60}")
print(f"{'Model':<25} {'7-class':<15} {'3-class':<15}")
print("-" * 55)
for name, label in models:
    a7 = f"{results_7[name]['acc']:.3f}±{results_7[name]['std']:.3f}"
    a3 = f"{results_3[name]['acc']:.3f}±{results_3[name]['std']:.3f}"
    print(f"  {label:<23} {a7:<15} {a3:<15}")
print(f"  {'Chance':<23} {'0.143':<15} {'0.333':<15}")

best_7 = max(results_7, key=lambda k: results_7[k]['acc'])
best_3 = max(results_3, key=lambda k: results_3[k]['acc'])
model_labels = dict(models)
print(f"\n★ Best 7-class: {model_labels[best_7]} ({results_7[best_7]['acc']:.3f})")
print(f"★ Best 3-class: {model_labels[best_3]} ({results_3[best_3]['acc']:.3f})")


BASELINE COMPARISON — All 1664 features

  Logistic Regression (7-class)... Acc=0.148±0.022  F1=0.145
  Logistic Regression (3-class)... Acc=0.342±0.014  F1=0.351

  SVM (RBF) (7-class)... Acc=0.133±0.013  F1=0.127
  SVM (RBF) (3-class)... Acc=0.324±0.024  F1=0.321

  Random Forest (7-class)... Acc=0.183±0.006  F1=0.142
  Random Forest (3-class)... Acc=0.489±0.010  F1=0.340

  Gradient Boosting (7-class)... Acc=0.156±0.022  F1=0.140
  Gradient Boosting (3-class)... Acc=0.443±0.018  F1=0.373

Model                     7-class         3-class        
-------------------------------------------------------
  Logistic Regression     0.148±0.022     0.342±0.014    
  SVM (RBF)               0.133±0.013     0.324±0.024    
  Random Forest           0.183±0.006     0.489±0.010    
  Gradient Boosting       0.156±0.022     0.443±0.018    
  Chance                  0.143           0.333          

★ Best 7-class: Random Forest (0.183)
★ Best 3-class: Random Forest (0.489)


In [6]:
# ═══════════════════════════════════════════════════════════
# CELL 3: EXPERIMENTAL PIPELINES
# ═══════════════════════════════════════════════════════════
# Test two techniques to handle the high-dimensional regime
# (1664 features / 2268 samples):
#   A. Feature Selection — keep only informative features
#   B. PCA — compress to 200 principal components

def evaluate_pipeline(X, y, groups, model_name, pipeline_name):
    """Same as evaluate_model but returns just acc, std."""
    acc, std, f1, f1_std = evaluate_model(X, y, groups, model_name)
    print(f"    {pipeline_name} + {model_labels.get(model_name, model_name)}: "
          f"Acc={acc:.3f}±{std:.3f}  F1={f1:.3f}")
    return acc


# ── A. Feature Selection ──
# SelectFromModel keeps features with importance > median.
# This removes noise features that hurt generalization.
print("=" * 60)
print("EXPERIMENT A: Feature Selection (SelectFromModel)")
print("=" * 60)

selector = SelectFromModel(
    RandomForestClassifier(n_estimators=200, class_weight='balanced',
                           random_state=42, n_jobs=-1),
    threshold='median'
)
selector.fit(X, y_7_encoded)
X_selected = selector.transform(X)
print(f"\n  Features: {X.shape[1]} → {X_selected.shape[1]} "
      f"({X_selected.shape[1]/X.shape[1]:.0%} retained)")

# Test best baseline model on selected features
print(f"\n  7-class results:")
sel_7_rf = evaluate_pipeline(X_selected, y_7_encoded, groups, 'rf', 'FeatureSelection')
sel_7_gb = evaluate_pipeline(X_selected, y_7_encoded, groups, 'gb', 'FeatureSelection')
print(f"  3-class results:")
sel_3_rf = evaluate_pipeline(X_selected, y_3_encoded, groups, 'rf', 'FeatureSelection')
sel_3_gb = evaluate_pipeline(X_selected, y_3_encoded, groups, 'gb', 'FeatureSelection')


# ── B. PCA Compression ──
# Reduce 1664 features to 200 principal components.
# Captures dominant variance while removing noise.
print(f"\n{'='*60}")
print("EXPERIMENT B: PCA (200 components)")
print("=" * 60)

scaler_pca = StandardScaler()
X_scaled_pca = scaler_pca.fit_transform(X)
pca = PCA(n_components=200, random_state=42)
X_pca = pca.fit_transform(X_scaled_pca)
print(f"\n  Variance explained: {pca.explained_variance_ratio_.sum():.1%}")
print(f"  Features: {X.shape[1]} → {X_pca.shape[1]}")

print(f"\n  7-class results:")
pca_7_rf = evaluate_pipeline(X_pca, y_7_encoded, groups, 'rf', 'PCA(200)')
pca_7_gb = evaluate_pipeline(X_pca, y_7_encoded, groups, 'gb', 'PCA(200)')
print(f"  3-class results:")
pca_3_rf = evaluate_pipeline(X_pca, y_3_encoded, groups, 'rf', 'PCA(200)')
pca_3_gb = evaluate_pipeline(X_pca, y_3_encoded, groups, 'gb', 'PCA(200)')


# ── Summary ──
print(f"\n{'='*60}")
print("EXPERIMENT SUMMARY (7-class accuracy)")
print("=" * 60)
print(f"  {'Pipeline':<30} {'RF':<10} {'GB':<10}")
print(f"  {'-'*50}")
print(f"  {'All 1664 features':<30} {results_7['rf']['acc']:<10.3f} {results_7['gb']['acc']:<10.3f}")
print(f"  {'Feature Selection':<30} {sel_7_rf:<10.3f} {sel_7_gb:<10.3f}")
print(f"  {'PCA(200)':<30} {pca_7_rf:<10.3f} {pca_7_gb:<10.3f}")

EXPERIMENT A: Feature Selection (SelectFromModel)

  Features: 1664 → 832 (50% retained)

  7-class results:
    FeatureSelection + Random Forest: Acc=0.191±0.014  F1=0.150
    FeatureSelection + Gradient Boosting: Acc=0.159±0.010  F1=0.144
  3-class results:
    FeatureSelection + Random Forest: Acc=0.493±0.014  F1=0.344
    FeatureSelection + Gradient Boosting: Acc=0.452±0.011  F1=0.391

EXPERIMENT B: PCA (200 components)

  Variance explained: 95.8%
  Features: 1664 → 200

  7-class results:
    PCA(200) + Random Forest: Acc=0.172±0.013  F1=0.122
    PCA(200) + Gradient Boosting: Acc=0.162±0.016  F1=0.134
  3-class results:
    PCA(200) + Random Forest: Acc=0.495±0.012  F1=0.332
    PCA(200) + Gradient Boosting: Acc=0.460±0.025  F1=0.380

EXPERIMENT SUMMARY (7-class accuracy)
  Pipeline                       RF         GB        
  --------------------------------------------------
  All 1664 features              0.183      0.156     
  Feature Selection              0.191      0.1

In [7]:
# ═══════════════════════════════════════════════════════════
# CELL 4: FEATURE IMPORTANCE ANALYSIS
# ═══════════════════════════════════════════════════════════
# Which brain regions and frequency bands matter most for
# emotion detection? This adds interpretability.

print("=" * 60)
print("FEATURE IMPORTANCE — Top 30 Features")
print("=" * 60)

# Train RF on all data for importance extraction
scaler_imp = StandardScaler()
X_imp = scaler_imp.fit_transform(X)
rf_imp = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                 random_state=42, n_jobs=-1)
rf_imp.fit(X_imp, y_7_encoded)

importances = rf_imp.feature_importances_
top_30_idx = np.argsort(importances)[-30:][::-1]

# Build human-readable feature names
# First 832 = DE features, next 832 = PSD features
# DE: 640 band stats (64ch × 5bands × 2stats) + 192 ratios (64ch × 3ratios)
bands = ['delta', 'theta', 'alpha', 'beta', 'gamma']
stats = ['mean', 'std']
ratios = ['alpha_beta', 'theta_beta', 'gamma_alpha']

def feature_name(idx):
    feat_type = 'DE' if idx < 832 else 'PSD'
    local_idx = idx % 832
    if local_idx < 640:  # band stats
        ch = local_idx // 10
        remainder = local_idx % 10
        band = remainder // 2
        stat = remainder % 2
        return f"{feat_type}_ch{ch:02d}_{bands[band]}_{stats[stat]}"
    else:  # ratios
        ratio_idx = local_idx - 640
        ch = ratio_idx // 3
        ratio = ratio_idx % 3
        return f"{feat_type}_ch{ch:02d}_{ratios[ratio]}"

print(f"\n  {'Rank':<6} {'Feature':<35} {'Importance':<10}")
print(f"  {'-'*51}")
for rank, idx in enumerate(top_30_idx, 1):
    name = feature_name(idx)
    print(f"  {rank:<6} {name:<35} {importances[idx]:.4f}")

# Band summary
print(f"\n{'='*60}")
print("BAND IMPORTANCE SUMMARY")
print("=" * 60)
band_importance = {b: 0.0 for b in bands + ratios}
for idx, imp in enumerate(importances):
    name = feature_name(idx)
    for b in bands + ratios:
        if b in name:
            band_importance[b] += imp
            break

for b, imp in sorted(band_importance.items(), key=lambda x: -x[1]):
    bar = '█' * int(imp * 200)
    print(f"  {b:<15} {imp:.4f} {bar}")


FEATURE IMPORTANCE — Top 30 Features

  Rank   Feature                             Importance
  ---------------------------------------------------
  1      PSD_ch25_delta_mean                 0.0011
  2      PSD_ch56_theta_beta                 0.0010
  3      PSD_ch15_delta_mean                 0.0010
  4      PSD_ch16_delta_mean                 0.0010
  5      PSD_ch06_theta_mean                 0.0010
  6      DE_ch05_theta_std                   0.0010
  7      DE_ch02_theta_std                   0.0010
  8      PSD_ch03_delta_mean                 0.0010
  9      DE_ch42_theta_std                   0.0010
  10     PSD_ch36_theta_beta                 0.0010
  11     PSD_ch51_delta_mean                 0.0010
  12     PSD_ch48_delta_mean                 0.0010
  13     PSD_ch21_theta_mean                 0.0010
  14     DE_ch57_alpha_std                   0.0010
  15     DE_ch56_beta_std                    0.0010
  16     DE_ch00_theta_std                   0.0010
  17     DE_ch23_the

In [8]:
# ═══════════════════════════════════════════════════════════
# CELL 5: SAVE BEST MODEL WITH CALIBRATION
# ═══════════════════════════════════════════════════════════
# Trains the best model on all data, wraps it with
# CalibratedClassifierCV so predict_proba outputs real
# probabilities (not just RF vote fractions).

# Determine overall best pipeline from all experiments
all_results = {
    'RF (all features)': results_7['rf']['acc'],
    'GB (all features)': results_7['gb']['acc'],
    'RF (feat select)': sel_7_rf,
    'GB (feat select)': sel_7_gb,
    'RF (PCA 200)': pca_7_rf,
    'GB (PCA 200)': pca_7_gb,
}
best_pipeline = max(all_results, key=all_results.get)
print(f"Best overall pipeline: {best_pipeline} (acc={all_results[best_pipeline]:.3f})")
print(f"\nAll results:")
for name, acc in sorted(all_results.items(), key=lambda x: -x[1]):
    marker = ' ★' if name == best_pipeline else ''
    print(f"  {name:<25} {acc:.3f}{marker}")

# ── Train final model (all features, RF — default safe choice) ──
# Using all 1664 features because:
# 1. The backend EEG simulator generates exactly 1664 features
# 2. Feature selection / PCA would require saving extra transformers
#    and updating the simulator to match
# The experiments above show if there's a significant gain from
# compression — you can switch later if needed.

print(f"\nTraining final RF model on all 1664 features (for backend compatibility)...")

scaler_final = StandardScaler()
X_final = scaler_final.fit_transform(X)

# Train base model
base_model = RandomForestClassifier(
    n_estimators=200, class_weight='balanced',
    random_state=42, n_jobs=-1
)
base_model.fit(X_final, y_7_encoded)

# Calibrate probabilities using 5-fold CV
# This makes predict_proba output real probabilities,
# not just vote fractions — critical for dynamic confidence.
print("Calibrating probabilities (isotonic regression)...")
calibrated_model = CalibratedClassifierCV(
    base_model, method='isotonic', cv=5
)
calibrated_model.fit(X_final, y_7_encoded)

# Verify calibration improved probability estimates
raw_proba = base_model.predict_proba(X_final)
cal_proba = calibrated_model.predict_proba(X_final)
print(f"\nProbability comparison (sample 0):")
print(f"  Raw RF:     {[f'{p:.3f}' for p in raw_proba[0]]}")
print(f"  Calibrated: {[f'{p:.3f}' for p in cal_proba[0]]}")

# Save artifacts
joblib.dump(calibrated_model, "eeg_emotion_model.pkl")
joblib.dump(scaler_final, "eeg_scaler.pkl")
joblib.dump(le_7, "eeg_label_encoder_7.pkl")
joblib.dump(le_3, "eeg_label_encoder_3.pkl")

print(f"\n✅ Saved artifacts:")


Best overall pipeline: RF (feat select) (acc=0.191)

All results:
  RF (feat select)          0.191 ★
  RF (all features)         0.183
  RF (PCA 200)              0.172
  GB (PCA 200)              0.162
  GB (feat select)          0.159
  GB (all features)         0.156

Training final RF model on all 1664 features (for backend compatibility)...
Calibrating probabilities (isotonic regression)...

Probability comparison (sample 0):
  Raw RF:     ['0.040', '0.055', '0.020', '0.140', '0.040', '0.010', '0.695']
  Calibrated: ['0.016', '0.076', '0.035', '0.272', '0.200', '0.090', '0.311']

✅ Saved artifacts:
